In [1]:
from pyspark.sql import SparkSession

In [2]:
import pandas as pd
pd.read_csv('test1.csv')#.info()

,Name,age,Experience
0,Krish,31,10
1,Sudhanshu,30,8
2,Sunny,29,4


In [4]:
spark = SparkSession.builder.appName("Practice").getOrCreate()

In [5]:
spark

In [6]:
## Read the Data

df_pyspark = spark.read.csv('test1.csv', header=True, inferSchema=True)
# df_pyspark = spark.read.option('header', 'true').csv('test1.csv', inferSchema=True) # Alternative way

In [8]:
df_pyspark.show()

+---------+---+----------+
|     Name|age|Experience|
+---------+---+----------+
|    Krish| 31|        10|
|Sudhanshu| 30|         8|
|    Sunny| 29|         4|
+---------+---+----------+



In [9]:
type(df_pyspark)

pyspark.sql.dataframe.DataFrame

In [10]:
# printSchema = df.info

df_pyspark.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)



In [12]:
df_pyspark.columns

['Name', 'age', 'Experience']

In [13]:
df_pyspark.head(3)

[Row(Name='Krish', age=31, Experience=10),
 Row(Name='Sudhanshu', age=30, Experience=8),
 Row(Name='Sunny', age=29, Experience=4)]

In [14]:
# df_pyspark.select('Name').show()
df_pyspark.select(['Name', 'Experience']).show()

+---------+----------+
|     Name|Experience|
+---------+----------+
|    Krish|        10|
|Sudhanshu|         8|
|    Sunny|         4|
+---------+----------+



In [15]:
df_pyspark.dtypes

[('Name', 'string'), ('age', 'int'), ('Experience', 'int')]

In [16]:
# Describe DF

df_pyspark.describe().show()

+-------+-----+----+-----------------+
|summary| Name| age|       Experience|
+-------+-----+----+-----------------+
|  count|    3|   3|                3|
|   mean| null|30.0|7.333333333333333|
| stddev| null| 1.0|3.055050463303893|
|    min|Krish|  29|                4|
|    max|Sunny|  31|               10|
+-------+-----+----+-----------------+



In [17]:
# Adding columns

df_pyspark = df_pyspark.withColumn('Experience After 2 Years', df_pyspark['Experience'] + 2)
df_pyspark.show()

+---------+---+----------+------------------------+
|     Name|age|Experience|Experience After 2 Years|
+---------+---+----------+------------------------+
|    Krish| 31|        10|                      12|
|Sudhanshu| 30|         8|                      10|
|    Sunny| 29|         4|                       6|
+---------+---+----------+------------------------+



In [18]:
# Dropping columns

df_pyspark = df_pyspark.drop('Experience After 2 Years')
df_pyspark.show()

+---------+---+----------+
|     Name|age|Experience|
+---------+---+----------+
|    Krish| 31|        10|
|Sudhanshu| 30|         8|
|    Sunny| 29|         4|
+---------+---+----------+



In [19]:
# Renaming columns

df_pyspark.withColumnRenamed('Name', 'New Name').show()

+---------+---+----------+
| New Name|age|Experience|
+---------+---+----------+
|    Krish| 31|        10|
|Sudhanshu| 30|         8|
|    Sunny| 29|         4|
+---------+---+----------+



In [35]:
df_pyspark2 = spark.read.csv('test2.csv', header=True)
df_pyspark2.show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [32]:
df_pyspark2.dtypes

[('Name', 'string'), ('age', 'int'), ('Experience', 'int'), ('Salary', 'int')]

In [21]:
# Drop rows based on the null values

# df_pyspark2.na.drop().show()
# df_pyspark2.na.drop(how='any').show()
#df_pyspark2.na.drop(how='any', thresh=2).show() # how many tolerable null values
df_pyspark2.na.drop(how='any', subset=['Experience']).show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
|     null| 34|        10| 38000|
+---------+---+----------+------+



In [34]:
## Filling the missing values

df_pyspark2.na.fill('Missing Value', ['Experience', 'age']).show()

+---------+-------------+-------------+------+
|     Name|          age|   Experience|Salary|
+---------+-------------+-------------+------+
|    Krish|           31|           10| 30000|
|Sudhanshu|           30|            8| 25000|
|    Sunny|           29|            4| 20000|
|     Paul|           24|            3| 20000|
|   Harsha|           21|            1| 15000|
|  Shubham|           23|            2| 18000|
|   Mahesh|Missing Value|Missing Value| 40000|
|     null|           34|           10| 38000|
|     null|           36|Missing Value|  null|
+---------+-------------+-------------+------+



In [89]:
## Impute mean to null values

from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols=['age', 'Experience', 'Salary'], 
    outputCols=[f"{c}_imputed" for c in ['age', 'Experience', 'Salary']]
    ).setStrategy("median")

# Add imputation cols to df
imputer.fit(df_pyspark2).transform(df_pyspark2).show()

+---------+----+----------+------+-----------+------------------+--------------+
|     Name| age|Experience|Salary|age_imputed|Experience_imputed|Salary_imputed|
+---------+----+----------+------+-----------+------------------+--------------+
|    Krish|  31|        10| 30000|         31|                10|         30000|
|Sudhanshu|  30|         8| 25000|         30|                 8|         25000|
|    Sunny|  29|         4| 20000|         29|                 4|         20000|
|     Paul|  24|         3| 20000|         24|                 3|         20000|
|   Harsha|  21|         1| 15000|         21|                 1|         15000|
|  Shubham|  23|         2| 18000|         23|                 2|         18000|
|   Mahesh|null|      null| 40000|         29|                 4|         40000|
|     null|  34|        10| 38000|         34|                10|         38000|
|     null|  36|      null|  null|         36|                 4|         20000|
+---------+----+----------+-

## Filtering

In [38]:
df_pyspark3 = spark.read.csv('test2.csv', header=True, inferSchema=True)
df_pyspark3.show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [39]:
df_pyspark3 = df_pyspark3.na.drop(how='any')
df_pyspark3.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [43]:
# Salary of the people who earn less than or equal to 20000

df_pyspark3.filter("Salary<=20000").show()
# df_pyspark3.filter(df_pyspark3["Salary"]<=20000).show() --> Alternative way

+-------+---+----------+------+
|   Name|age|Experience|Salary|
+-------+---+----------+------+
|  Sunny| 29|         4| 20000|
|   Paul| 24|         3| 20000|
| Harsha| 21|         1| 15000|
|Shubham| 23|         2| 18000|
+-------+---+----------+------+



In [41]:
df_pyspark3.filter("Salary<=20000").select(["Name", "Age"]).show()

+-------+---+
|   Name|Age|
+-------+---+
|  Sunny| 29|
|   Paul| 24|
| Harsha| 21|
|Shubham| 23|
+-------+---+



In [46]:
df_pyspark3.filter((df_pyspark3["Salary"]<=20000) & 
                   (df_pyspark3["Salary"]>=15000)).show()

+-------+---+----------+------+
|   Name|age|Experience|Salary|
+-------+---+----------+------+
|  Sunny| 29|         4| 20000|
|   Paul| 24|         3| 20000|
| Harsha| 21|         1| 15000|
|Shubham| 23|         2| 18000|
+-------+---+----------+------+



In [47]:
df_pyspark3.filter(~(df_pyspark3["Salary"]<=20000)).show() 

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
+---------+---+----------+------+



## Groupby and Aggregate Functions

In [48]:
df_pyspark4 = spark.read.csv('test3.csv', header=True, inferSchema=True)
df_pyspark4.show()

+---------+------------+------+
|     Name| Departments|salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|    Krish|         IOT|  5000|
|   Mahesh|    Big Data|  4000|
|    Krish|    Big Data|  4000|
|   Mahesh|Data Science|  3000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|         IOT| 10000|
|Sudhanshu|    Big Data|  5000|
|    Sunny|Data Science| 10000|
|    Sunny|    Big Data|  2000|
+---------+------------+------+



In [49]:
df_pyspark4.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Departments: string (nullable = true)
 |-- salary: integer (nullable = true)



In [56]:
# Groupby

df_pyspark4.groupBy('Name').sum().show()

+---------+-----------+
|     Name|sum(salary)|
+---------+-----------+
|Sudhanshu|      35000|
|    Sunny|      12000|
|    Krish|      19000|
|   Mahesh|       7000|
+---------+-----------+



In [60]:
df_pyspark4.groupBy('Departments').count().show()

+------------+-----+
| Departments|count|
+------------+-----+
|         IOT|    2|
|    Big Data|    4|
|Data Science|    4|
+------------+-----+



In [61]:
# Direct aggregate function

df_pyspark4.agg({'Salary':'sum'}).show()

+-----------+
|sum(Salary)|
+-----------+
|      73000|
+-----------+



## PySpark ML

In [101]:
training = spark.read.csv('test2.csv', header=True, inferSchema=True)
training = training.na.drop(how='any')
training.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [89]:
training.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [90]:
training.columns

['Name', 'age', 'Experience', 'Salary']

In [91]:
from pyspark.ml.feature import VectorAssembler

feature_assembler = VectorAssembler(inputCols=["age", "Experience"], outputCol="Independent Features")

In [92]:
output = feature_assembler.transform(training)

In [93]:
output.show()

+---------+---+----------+------+--------------------+
|     Name|age|Experience|Salary|Independent Features|
+---------+---+----------+------+--------------------+
|    Krish| 31|        10| 30000|         [31.0,10.0]|
|Sudhanshu| 30|         8| 25000|          [30.0,8.0]|
|    Sunny| 29|         4| 20000|          [29.0,4.0]|
|     Paul| 24|         3| 20000|          [24.0,3.0]|
|   Harsha| 21|         1| 15000|          [21.0,1.0]|
|  Shubham| 23|         2| 18000|          [23.0,2.0]|
+---------+---+----------+------+--------------------+



In [74]:
output.columns

['Name', 'age', 'Experience', 'Salary', 'Independent Features']

In [94]:
finalized_data = output.select(["Independent Features", "Salary"])

In [95]:
finalized_data.show()

+--------------------+------+
|Independent Features|Salary|
+--------------------+------+
|         [31.0,10.0]| 30000|
|          [30.0,8.0]| 25000|
|          [29.0,4.0]| 20000|
|          [24.0,3.0]| 20000|
|          [21.0,1.0]| 15000|
|          [23.0,2.0]| 18000|
+--------------------+------+



In [100]:
from pyspark.ml.regression import LinearRegression

# Train test split
train_data, test_data = finalized_data.randomSplit([0.75, 0.25])
regressor = LinearRegression(featuresCol="Independent Features", labelCol="Salary")
regressor = regressor.fit(train_data)

23/09/16 19:59:24 WARN Instrumentation: [57e98014] regParam is zero, which might cause numerical instability and overfitting.


In [97]:
regressor.coefficients

DenseVector([-706.4673, 3550.3761])

In [98]:
regressor.intercept

26291.855982270346

In [99]:
pred_results = regressor.evaluate(test_data)
pred_results.predictions.show()

+--------------------+------+------------------+
|Independent Features|Salary|        prediction|
+--------------------+------+------------------+
|          [23.0,2.0]| 18000|17143.861340872107|
|          [30.0,8.0]| 25000| 33500.84732581509|
|         [31.0,10.0]| 30000| 39895.13232959475|
+--------------------+------+------------------+



In [ ]:
# Performance metrics

pred_results.r2, pred_results.meanAbsoluteError, pred_results.meanSquaredError

-------

## Before Vector Assembler if applicable

In [ ]:
## Handling categorical features
from pyspark.ml.feature import StringIndexer # to convert string categorical features into numerical features

indexer = StringIndexer(inputCols=['sex', 'smoker', 'day', 'time'], outputCols=['sex_indexed', 'smoker_indexed', 'day_indexed', 'time_indexed'])
df_r = indexer.fit(df).transform(df)
display(df_r)

In [90]:
spark.stop()